# Step 1 — Build the Vector Database

This is the **first half** of our RAG (Retrieval-Augmented Generation) system.

**What is RAG?** Instead of asking an AI model a question and hoping it remembers the answer
from its training data, we first *search our own documents* for the relevant passages, and
then hand those passages to the AI model along with the question. This lets the model answer
using facts from a specific book/document, even if it never saw that document during training.

**What this notebook does, step by step:**
1. Read the text out of a PDF file.
2. Split that text into small chunks (a few sentences each).
3. Turn each chunk into an *embedding* (a list of numbers that represents its meaning).
4. Store the chunks + embeddings in **ChromaDB**, a local database built for this kind of search.

You only need to run this notebook **once**. After that, the database is saved to disk and
notebook `02_Ask_Questions.ipynb` can reuse it.

## 1. Install the libraries we need

If you haven't already, install the required packages (only needs to be done once per computer).
Run this in a terminal from the project folder:

```bash
pip install -r requirements.txt
```

## 2. Import libraries and load our secret API key

The OpenAI API key is stored in a `.env` file (in the project root) instead of being typed
directly into the notebook. This way the secret key never ends up saved inside notebook code.

In [1]:
import os
from pypdf import PdfReader
import chromadb
from openai import OpenAI
from dotenv import load_dotenv

# Reads the OPENAI_API_KEY variable out of the .env file and into the environment
load_dotenv("../.env")

# The OpenAI client automatically picks up OPENAI_API_KEY from the environment
client = OpenAI()

## 3. Settings

A few constants we'll reuse throughout the notebook. Keeping them together at the top makes
them easy to find and change later.

In [2]:
PDF_PATH = "../Data/[ML] Introduction to Machine Learning with Python (2017).pdf"

CHROMA_PATH = "../chroma_db"      # folder where the database is saved on disk
COLLECTION_NAME = "ml_book"       # the name we give our set of stored chunks

EMBEDDING_MODEL = "text-embedding-3-small"   # OpenAI model that turns text into numbers

CHUNK_SIZE_WORDS = 200   # how many words go in each chunk
CHUNK_OVERLAP_WORDS = 40 # how many words each chunk shares with the previous one

## 4. Read the text out of the PDF

We open the PDF and pull the plain text out of every page, keeping track of which page
number each bit of text came from (useful later for citing sources).

In [3]:
reader = PdfReader(PDF_PATH)
print(f"The PDF has {len(reader.pages)} pages.")

pages_text = []  # will hold one string of text per page
for page in reader.pages:
    text = page.extract_text() or ""
    pages_text.append(text)

# Peek at the text from the first page that actually has content
for i, text in enumerate(pages_text):
    if text.strip():
        print(f"\n--- Preview of page {i + 1} ---\n")
        print(text[:500])
        break

The PDF has 392 pages.



--- Preview of page 1 ---

Andreas C. Müller & Sarah Guido
Introduction to 
Machine 
Learning  
with P y t h o n   
A GUIDE FOR DATA SCIENTISTS



## 5. Split each page into small chunks

Why chunk at all? Two reasons:
- Embeddings work best on short, focused pieces of text rather than whole chapters.
- Later, we only want to hand the AI model the few paragraphs that actually answer the
  question, not the entire book.

We split each page's text into overlapping groups of words. The overlap means a sentence
that gets cut in half at a chunk boundary still appears in full inside the neighboring chunk.

In [4]:
def split_into_chunks(text, chunk_size=CHUNK_SIZE_WORDS, overlap=CHUNK_OVERLAP_WORDS):
    """Split `text` into a list of overlapping word-chunks."""
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap  # move forward, re-using the last `overlap` words

    return chunks

In [5]:
# Build one big list of chunks, remembering which page each chunk came from
all_chunks = []      # the text of every chunk
chunk_pages = []     # the page number that chunk came from

for page_number, text in enumerate(pages_text, start=1):
    if not text.strip():
        continue  # skip blank pages (e.g. images with no extractable text)

    page_chunks = split_into_chunks(text)
    all_chunks.extend(page_chunks)
    chunk_pages.extend([page_number] * len(page_chunks))

print(f"Created {len(all_chunks)} chunks from {len(reader.pages)} pages.")

Created 843 chunks from 392 pages.


## 6. Turn each chunk into an embedding

An **embedding** is a list of numbers (a vector) that represents the *meaning* of a piece of
text. Chunks with similar meaning end up with similar numbers. This is what lets us later
search by meaning instead of by exact keyword matching.

We send the chunks to OpenAI in batches (100 at a time) instead of one-by-one, since that's
much faster and uses fewer API calls.

In [6]:
def get_embeddings(texts, batch_size=100):
    """Return one embedding vector per input text, in the same order."""
    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)
        print(f"Embedded chunks {start + 1}-{start + len(batch)} of {len(texts)}")

    return all_embeddings

In [7]:
chunk_embeddings = get_embeddings(all_chunks)
print(f"\nEach embedding is a vector of {len(chunk_embeddings[0])} numbers.")

Embedded chunks 1-100 of 843


Embedded chunks 101-200 of 843


Embedded chunks 201-300 of 843


Embedded chunks 301-400 of 843


Embedded chunks 401-500 of 843


Embedded chunks 501-600 of 843


Embedded chunks 601-700 of 843


Embedded chunks 701-800 of 843


Embedded chunks 801-843 of 843

Each embedding is a vector of 1536 numbers.


## 7. Save everything into ChromaDB

ChromaDB stores, for every chunk: its text, its embedding, and some metadata (here, the page
number). `PersistentClient` writes the database to a folder on disk, so it's still there the
next time we open a notebook — we don't need to redo any of the steps above again.

In [8]:
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# get_or_create_collection: reuse the collection if it already exists, otherwise make a new one
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)

chunk_ids = [f"chunk-{i}" for i in range(len(all_chunks))]
chunk_metadata = [{"page": page, "source": PDF_PATH} for page in chunk_pages]

collection.add(
    ids=chunk_ids,
    documents=all_chunks,
    embeddings=chunk_embeddings,
    metadatas=chunk_metadata,
)

print(f"Saved {collection.count()} chunks to the '{COLLECTION_NAME}' collection.")

Saved 843 chunks to the 'ml_book' collection.


## 8. Quick sanity check

Let's make sure the database actually works by searching for a topic and seeing if we get
back sensible chunks. We embed a test question the same way we embedded the chunks, then ask
ChromaDB for the closest matches.

In [9]:
test_question = "What is supervised learning?"
test_embedding = get_embeddings([test_question])[0]

results = collection.query(query_embeddings=[test_embedding], n_results=3)

for rank, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0]), start=1):
    print(f"\n--- Match {rank} (page {meta['page']}) ---")
    print(doc[:300])

Embedded chunks 1-1 of 1

--- Match 1 (page 102) ---
Supervised Learning

--- Match 2 (page 116) ---
Chapter 2: Supervised Learning

--- Match 3 (page 39) ---
CHAPTER 2 Supervised Learning As we mentioned earlier, supervised machine learning is one of the most commonly used and successful types of machine learning. In this chapter, we will describe super‐ vised learning in more detail and explain several popular supervised learning algo‐ rithms. We alread


## 9. Three one-line checks

Quick, single-line looks at what actually got created — a raw chunk, the raw contents of the database, and the raw result of a retrieval query.

**Validate a chunk** — see exactly what one chunk of text looks like (this is the raw unit that gets embedded and stored):

In [10]:
f"Chunk from page {chunk_pages[100]}:\n\n{all_chunks[100]}"

'Chunk from page 44:\n\n4 Discussing all of them is beyond the scope of the book, and we refer you to the scikit-learn documentation for more details. view of how each algorithm builds a model. We will examine the strengths and weak‐ nesses of each algorithm, and what kind of data they can best be applied to. We will also explain the meaning of the most important parameters and options. 4 Many algo‐ rithms have a classification and a regression variant, and we will describe both. It is not necessary to read through the descriptions of each algorithm in detail, but understanding the models will give you a better feeling for the different ways machine learning algorithms can work. This chapter can also be used as a reference guide, and you can come back to it when you are unsure about the workings of any of the algorithms. Some Sample Datasets We will use several datasets to illustrate the different algorithms. Some of the datasets will be small and synthetic (meaning made-up), designed 

**Understand ChromaDB** — `peek()` returns the raw records stored in the database: their ids, document text, embedding vectors, and metadata:

In [11]:
collection.peek(limit=2)

{'ids': ['chunk-0', 'chunk-1'],
 'embeddings': array([[-0.02192688,  0.00847626, -0.01147461, ..., -0.02363586,
         -0.02522278, -0.02166748],
        [ 0.00295258, -0.0080719 ,  0.03210449, ..., -0.01696777,
         -0.04510498, -0.02455139]], shape=(2, 1536)),
 'documents': ['Andreas C. Müller & Sarah Guido Introduction to Machine Learning with P y t h o n A GUIDE FOR DATA SCIENTISTS',
  'Andreas C. Müller and Sarah Guido Introduction to Machine Learning with Python A Guide for Data Scientists Boston Farnham Sebastopol TokyoBeijing Boston Farnham Sebastopol TokyoBeijing'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'page': 1,
   'source': '../Data/[ML] Introduction to Machine Learning with Python (2017).pdf'},
  {'page': 3,
   'source': '../Data/[ML] Introduction to Machine Learning with Python (2017).pdf'}]}

**Understand retrieval** — the raw output of `collection.query()`: it ranks stored chunks by embedding similarity to our test question and returns them with a `distance` (lower = more similar):

In [12]:
collection.query(query_embeddings=[test_embedding], n_results=3, include=["documents", "metadatas", "distances"])

{'ids': [['chunk-223', 'chunk-249', 'chunk-87']],
 'embeddings': None,
 'documents': [['Supervised Learning',
   'Chapter 2: Supervised Learning',
   'CHAPTER 2 Supervised Learning As we mentioned earlier, supervised machine learning is one of the most commonly used and successful types of machine learning. In this chapter, we will describe super‐ vised learning in more detail and explain several popular supervised learning algo‐ rithms. We already saw an application of supervised machine learning in Chapter 1: classifying iris flowers into several species using physical measurements of the flowers. Remember that supervised learning is used whenever we want to predict a certain outcome from a given input, and we have examples of input/output pairs. We build a machine learning model from these input/output pairs, which comprise our training set. Our goal is to make accurate predictions for new, never-before-seen data. Super‐ vised learning often requires human effort to build the traini

The database is now built and saved in the `chroma_db/` folder.

Next: open **`02_Ask_Questions.ipynb`** to ask real questions about the book.